In [1]:
file = open('data.txt', 'w')
try:
    file.write("Hello World")
    1 / 0  # CRASH! An exception happens here!
finally:
    # We have to remember to write this finally block 
    # to guarantee the file closes.
    file.close()

ZeroDivisionError: division by zero

In [2]:
# The 'with' statement handles the setup and the guaranteed teardown automatically!
with open('data.txt', 'w') as file:
    file.write("Hello World")
    1 / 0 # CRASH! But Python still silently closes the file for us.

ZeroDivisionError: division by zero

In [3]:
from contextlib import contextmanager

@contextmanager
def my_timer():
    # --- This acts as __enter__ ---
    print("1. Starting the timer...")
    
    # --- This PAUSES the function and hands control to the 'with' block ---
    yield  
    
    # --- This acts as __exit__ (Runs when the 'with' block ends) ---
    print("3. Stopping the timer...")

# Using it:
with my_timer():
    print("2. Doing some hard work...")

1. Starting the timer...
2. Doing some hard work...
3. Stopping the timer...


In [5]:
def gcd_loop(m, n):
    while n != 0:
        # We are constantly mutating (changing) the state of m and n
        m, n = n, m % n 
    return m

g = gcd_loop(10, 20)
g

10

# Interpreter code for parsing

In [12]:
def tokenize(s):
    # 1. Pad every parenthesis with spaces
    # 2. Split the string by whitespace!
    return s.replace('(', ' ( ').replace(')', ' ) ').split()
t =tokenize('(+ 5 4)')
t

['(', '+', '5', '4', ')']

In [13]:
#convert our flat list into an Abstract Syntax Tree (AST)
def atom(token):
    try:
        return int(token)
    except ValueError:
        try:
            return float(token)
        except ValueError:
            return str(token)


def read_tokens(tokens):
    token = tokens.pop(0)
    if token == '(':
        L = []
        while tokens[0] != ')':
            L.append(read_tokens(tokens))
        tokens.pop(0)
        return L
    
    elif token == ')':
        raise SyntaxError(")")
    
    else:
        return atom(token)

read_tokens(t)

['+', 5, 4]

In [17]:
import operator

# 1. The Memory (Environment)
global_env = {
    '+': operator.add,  
    '-': operator.sub,
    '*': operator.mul,
    'x': 10             
}

# 2. The Brain (Evaluator)
def evaluate(ast, env):
    match ast:
        # --- RULE 1: SIMPLE DATA ---
        case str():
            return env[ast]  
        case int() | float():
            return ast       
            
        # --- RULE 2: QUOTE ---
        case ['quote', exp]:
            return exp

        # --- RULE 3: IF STATEMENT ---
        case ['if', test, consequence, alternative]:
            if evaluate(test, env):
                return evaluate(consequence, env)
            else:
                return evaluate(alternative, env)

        # --- RULE 4: DEFINE (Save to Memory) ---
        case ['define', name, exp]:
            env[name] = evaluate(exp, env)
            
        # --- RULE 5: LAMBDA (Create a Function) ---
        # (We didn't code this together, but this is how it creates functions!)
        case ['lambda', params, body]:
            # Returns a Python lambda function that evaluates the body 
            # using a new local environment
            return lambda *args: evaluate(body, dict(env, **dict(zip(params, args))))

        # --- RULE 6: FUNCTION CALLS (The Catch-All) ---
        # This MUST be at the bottom!
        case [op, *args]:
            proc = evaluate(op, env)
            vals = [evaluate(arg, env) for arg in args]
            return proc(*vals)

In [ ]:
def repl(prompt='scheme> '):
    print("Welcome to your custom Scheme Interpreter! )
    
    # Enter an infinite loop
    while True:
        try:
            # 1. READ: Get text from the user
            user_input = input(prompt)
            
            # 2. PARSE: Turn the text into an AST using the functions you wrote
            tokens = tokenize(user_input)
            ast = read_tokens(tokens)
            
            # 3. EVAL: Run the code through our brain
            result = evaluate(ast, global_env)
            
            # 4. PRINT: Show the user the answer
            if result is not None:
                print(result)
                
        except Exception as e:
            # If the user types bad code, don't crash the whole program!
            print(f"Error: {e}")

# Start the program!
if __name__ == '__main__':
    repl()

Welcome to your custom Scheme Interpreter! (Type Ctrl+C to exit)
Error: pop from empty list
Error: pop from empty list
Error: pop from empty list
Error: pop from empty list
Error: pop from empty list
Error: '5+4'
1
55
5555
Error: pop from empty list


When you put an else block at the end of a loop, it means: "Run this code ONLY if the loop finished naturally without hitting a break statement.

In [4]:
flavors = ['apple', 'banana', 'orange']

for flavor in flavors:
    if flavor == 'banana':
        print("Found the banana!")
        break
else:
    # This ONLY runs if the loop exhausts the whole list 
    # without ever hitting the 'break' keyword.
    print("No banana flavor found!")

Found the banana!


In [ ]:
try:
    # ONLY the dangerous code goes here
    data = fetch_data_from_internet()
except ConnectionError:
    print("Failed to connect.")
else:
    # This ONLY runs if the try block succeeds!
    print("Success! Saving data...")
    save_to_database(data)